# Notebook 05: Error Analysis and Stability

**Purpose:** Understand where and why the baseline model fails.

Run this notebook after `04_baseline_features_and_model.ipynb`.
The goal is to generate **at least 3 actionable hypotheses** for Stage 3 feature engineering.

Key questions:
- Which time periods are worst for the baseline?
- Which lag horizons are most/least predictable?
- Are there systematic patterns in the residuals?

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src.data.load_data import load_train, load_labels, load_target_pairs, get_labels_for_lag
from src.evaluation.metric import spearman_sharpe_score, per_period_spearman

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print('Re-running baseline from notebook 04 to get predictions...')
print('(Factored helper functions to be moved to src/ in Stage 3)')

In [ ]:
# --- Reproduce the baseline split and predictions ---
# (Inline here until this logic is moved to src/models/baseline.py in Stage 3)

train  = load_train()
labels = load_labels()
pairs  = load_target_pairs()

feature_cols = [c for c in train.columns if c != 'date_id']
HOLDOUT_FRACTION = 0.20
MAX_LAG = 10

# Lag features
frames = []
for lag in [1, 2, 3, 5, 10]:
    shifted = train[feature_cols].shift(lag)
    shifted.columns = [f'{c}_lag{lag}' for c in feature_cols]
    frames.append(shifted)
X_df = pd.concat(frames, axis=1)
X_df.insert(0, 'date_id', train['date_id'].values)

date_ids = X_df['date_id'].values
X = X_df.drop(columns='date_id').values

T_total  = len(date_ids)
val_size = int(T_total * HOLDOUT_FRACTION)
split_id = date_ids[T_total - val_size]

valid_rows  = ~np.all(np.isnan(X), axis=1)
train_mask  = valid_rows & (date_ids < split_id)
val_mask    = valid_rows & (date_ids >= split_id)

col_means   = np.nanmean(X[train_mask], axis=0)

def impute(arr, means):
    out = arr.copy()
    idx = np.where(np.isnan(out))
    out[idx] = means[idx[1]]
    return out

X_tr_imp = impute(X[train_mask], col_means)
X_vl_imp = impute(X[val_mask],   col_means)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_imp)
X_vl_sc = scaler.transform(X_vl_imp)

print(f'Baseline reproduced: {train_mask.sum()} train, {val_mask.sum()} val rows.')

## 1. Per-lag performance breakdown

In [ ]:
lag_scores = {}

for lag in [1, 2, 3, 4]:
    lag_labels = get_labels_for_lag(labels, pairs, lag)
    target_cols = [c for c in lag_labels.columns if c != 'date_id']
    label_vals  = lag_labels.set_index('date_id')[target_cols]

    y_tr = label_vals.loc[date_ids[train_mask]].values
    y_vl = label_vals.loc[date_ids[val_mask]].values

    y_tr_imp = np.where(np.isnan(y_tr), 0.0, y_tr)

    model = Ridge(alpha=1.0)
    model.fit(X_tr_sc, y_tr_imp)
    y_pred = model.predict(X_vl_sc)

    y_pred_df = pd.DataFrame(y_pred,  index=date_ids[val_mask], columns=target_cols)
    y_true_df = pd.DataFrame(y_vl,    index=date_ids[val_mask], columns=target_cols)

    score = spearman_sharpe_score(y_pred_df, y_true_df, nan_policy='ignore')
    lag_scores[lag] = {'score': score, 'pred': y_pred_df, 'true': y_true_df}

    print(f'  Lag {lag}: Spearman-Sharpe = {score:.4f}')

print()
print('Best lag:', min(lag_scores, key=lambda k: -lag_scores[k]['score']))

In [ ]:
# Plot per-period Spearman for each lag
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True, sharey=True)

for i, lag in enumerate([1, 2, 3, 4]):
    ax = axes[i // 2][i % 2]
    pp = per_period_spearman(lag_scores[lag]['pred'], lag_scores[lag]['true'])
    pp.plot(ax=ax, alpha=0.7)
    ax.axhline(pp.mean(), color='red', lw=1, linestyle='--')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(f'Lag {lag}  (SS={lag_scores[lag]["score"]:.3f})')
    ax.set_xlabel('date_id')

fig.suptitle('Per-period Spearman by lag horizon (Ridge baseline, validation set)', fontsize=11)
plt.tight_layout()
plt.show()

## 2. Feature coefficient analysis

In [ ]:
# Use lag=1 model coefficients for interpretation
lag1_labels = get_labels_for_lag(labels, pairs, lag=1)
target_cols = [c for c in lag1_labels.columns if c != 'date_id']
y_tr = lag1_labels.set_index('date_id')[target_cols].loc[date_ids[train_mask]].values
y_tr_imp = np.where(np.isnan(y_tr), 0.0, y_tr)

model_lag1 = Ridge(alpha=1.0)
model_lag1.fit(X_tr_sc, y_tr_imp)

lag_feature_names = X_df.drop(columns='date_id').columns.tolist()

# Mean absolute coefficient across targets — proxy for overall feature importance
mean_abs_coef = np.abs(model_lag1.coef_).mean(axis=0)
coef_series = pd.Series(mean_abs_coef, index=lag_feature_names)
top_features = coef_series.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 5))
top_features.plot(kind='barh')
plt.xlabel('Mean |coefficient|')
plt.title('Top 20 features by mean absolute Ridge coefficient (lag=1 targets)')
plt.tight_layout()
plt.show()

## 3. Actionable hypotheses for Stage 3

Fill in after running this notebook:

1. **[TBD]**: Based on per-lag performance — e.g. "Lag 1 is most predictable; use separate models per lag"
2. **[TBD]**: Based on feature importance — e.g. "FX lag-1 features dominate; add FX momentum features"
3. **[TBD]**: Based on per-period plot — e.g. "Performance collapses in date_id 1700–1800; investigate regime change"

These hypotheses drive the feature engineering roadmap in `docs/stage2_plan.md`.